# EDA & Feature Engineering
**IEEE-CIS Fraud Detection — Vesta Corporation**

The dataset ships as two tables: 590k transactions and 144k identity records joined on `TransactionID`. This notebook merges them, audits nulls, visualises the main patterns, and engineers 10 behavioural features before handing off to the model notebook.

Outputs are written to `../outputs/` for use in notebooks 2 and 3.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.3f}'.format)

import os
_HERE    = os.path.dirname(os.path.abspath('__file__'))
DATA_DIR = os.path.join(_HERE, '..', 'data', 'ieee-fraud-detection') + os.sep
OUT_DIR  = os.path.join(_HERE, '..', 'outputs') + os.sep
os.makedirs(OUT_DIR, exist_ok=True)

## Load & Merge

Left join on `TransactionID` — not all transactions have identity records, and that's fine.

In [ ]:
print("Loading data...")

train_txn = pd.read_csv(DATA_DIR + 'train_transaction.csv')
train_id  = pd.read_csv(DATA_DIR + 'train_identity.csv')
test_txn  = pd.read_csv(DATA_DIR + 'test_transaction.csv')
test_id   = pd.read_csv(DATA_DIR + 'test_identity.csv')

print(f"Train transactions : {train_txn.shape}")
print(f"Train identity     : {train_id.shape}")
print(f"Test  transactions : {test_txn.shape}")
print(f"Test  identity     : {test_id.shape}")

In [ ]:
train = train_txn.merge(train_id, on='TransactionID', how='left')
test  = test_txn.merge(test_id,   on='TransactionID', how='left')

print(f"Merged train : {train.shape}")
print(f"Merged test  : {test.shape}")

## Class Distribution

3.5% fraud rate. This rules out accuracy as a useful metric — a model that predicts *legitimate* every time scores 96.5%. Everything downstream uses KS, Gini, and AUC-PR instead.

In [ ]:
fraud_rate = train['isFraud'].mean()
fraud_n    = train['isFraud'].sum()

print(f"Total transactions : {len(train):,}")
print(f"Fraud cases        : {fraud_n:,}")
print(f"Legitimate cases   : {len(train) - fraud_n:,}")
print(f"Fraud rate         : {fraud_rate*100:.2f}%")
print(f"Class imbalance    : 1 : {round((1-fraud_rate)/fraud_rate)}")

## Exploratory Analysis

In [ ]:
fig = plt.figure(figsize=(16, 12))
fig.suptitle('IEEE-CIS Fraud Detection — Exploratory Analysis', fontsize=14, y=0.98)
gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.4, wspace=0.35)

# Class imbalance
ax1 = fig.add_subplot(gs[0, 0])
counts = train['isFraud'].value_counts()
bars = ax1.bar(['Legitimate', 'Fraud'], counts.values,
               color=['#1D9E75', '#D85A30'], width=0.5)
ax1.set_title('Class distribution', fontsize=11)
ax1.set_ylabel('Count')
for bar, count in zip(bars, counts.values):
    ax1.text(bar.get_x()+bar.get_width()/2, bar.get_height()*1.01,
             f'{count:,}', ha='center', fontsize=9)

# Transaction amount by fraud
ax2 = fig.add_subplot(gs[0, 1])
legit_amt = train[train['isFraud']==0]['TransactionAmt'].clip(upper=1000)
fraud_amt = train[train['isFraud']==1]['TransactionAmt'].clip(upper=1000)
ax2.hist(legit_amt, bins=50, alpha=0.6, color='#1D9E75', label='Legit',  density=True)
ax2.hist(fraud_amt, bins=50, alpha=0.6, color='#D85A30', label='Fraud', density=True)
ax2.set_title('Transaction amount distribution', fontsize=11)
ax2.set_xlabel('Amount (USD, clipped at $1000)')
ax2.legend(fontsize=9)

# Fraud rate by ProductCD
ax3 = fig.add_subplot(gs[0, 2])
prod = train.groupby('ProductCD')['isFraud'].mean().sort_values(ascending=False)
bars3 = ax3.bar(prod.index, prod.values*100, color='#534AB7', width=0.5)
ax3.set_title('Fraud rate by product code', fontsize=11)
ax3.set_ylabel('Fraud rate (%)')
for bar, val in zip(bars3, prod.values*100):
    ax3.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.1,
             f'{val:.1f}%', ha='center', fontsize=8)

# Fraud rate by card network
ax4 = fig.add_subplot(gs[1, 0])
card = train.groupby('card4')['isFraud'].mean().dropna().sort_values(ascending=False)
ax4.barh(card.index, card.values*100, color='#534AB7')
ax4.set_title('Fraud rate by card network', fontsize=11)
ax4.set_xlabel('Fraud rate (%)')

# Fraud rate by card type (debit/credit)
ax5 = fig.add_subplot(gs[1, 1])
c6 = train.groupby('card6')['isFraud'].mean().dropna().sort_values(ascending=False)
ax5.bar(c6.index, c6.values*100,
        color=['#D85A30','#1D9E75','#534AB7','#BA7517'][:len(c6)], width=0.4)
ax5.set_title('Fraud rate by card type', fontsize=11)
ax5.set_ylabel('Fraud rate (%)')

# Fraud rate by hour of day
ax6 = fig.add_subplot(gs[1, 2])
train['hour'] = (train['TransactionDT'] // 3600) % 24
hour_fraud = train.groupby('hour')['isFraud'].mean()
ax6.plot(hour_fraud.index, hour_fraud.values*100, color='#D85A30', linewidth=2)
ax6.fill_between(hour_fraud.index, hour_fraud.values*100, alpha=0.15, color='#D85A30')
ax6.set_title('Fraud rate by hour of day', fontsize=11)
ax6.set_xlabel('Hour (UTC)')
ax6.set_ylabel('Fraud rate (%)')

plt.savefig(OUT_DIR + '01_eda_overview.png', dpi=150, bbox_inches='tight')
plt.show()

## Null Handling

A lot of the V-columns are >90% null. These get dropped — there's not enough signal to justify imputing them, and they'd add noise to the tree splits.

In [ ]:
null_pct  = (train.isnull().sum() / len(train) * 100).sort_values(ascending=False)
drop_cols = null_pct[null_pct > 90].index.tolist()

print(f"Columns with >90% nulls: {len(drop_cols)}")
print(f"Dropping: {drop_cols[:10]} ... ({len(drop_cols)} total)")

train.drop(columns=drop_cols, inplace=True)
test.drop(columns=[c for c in drop_cols if c in test.columns], inplace=True)

print(f"\nDataset after drop: {train.shape}")

## Feature Engineering

10 behavioural features built from domain logic. The V-columns in the raw data are anonymised, so these domain features give the model interpretable signals to anchor on — particularly important for SHAP explanations later.

Key ones:
- **`amt_zscore`** — how unusual is this amount *for this specific card*. A \$200 charge is normal for one card, suspicious for another.
- **`card1_txn_count`** — velocity proxy. Fraud often involves multiple rapid charges before the card is blocked.
- **`email_mismatch`** — purchaser and recipient email domains differ. Common in account takeover.

In [ ]:
def engineer_features(df):
    df = df.copy()

    df['amt_log'] = np.log1p(df['TransactionAmt'])

    card_stats = df.groupby('card1')['TransactionAmt'].agg(['mean','std']).reset_index()
    card_stats.columns = ['card1','card1_amt_mean','card1_amt_std']
    df = df.merge(card_stats, on='card1', how='left')
    df['amt_zscore'] = (
        (df['TransactionAmt'] - df['card1_amt_mean'])
        / df['card1_amt_std'].replace(0, np.nan)
    ).fillna(0)

    df['email_mismatch'] = (
        df['P_emaildomain'].fillna('') != df['R_emaildomain'].fillna('')
    ).astype(int)

    risky_domains = ['gmail.com','yahoo.com','hotmail.com','anonymous.com','']
    df['P_email_risky'] = df['P_emaildomain'].fillna('').isin(risky_domains).astype(int)

    df['hour']     = (df['TransactionDT'] // 3600) % 24
    df['is_night'] = ((df['hour'] >= 0) & (df['hour'] < 6)).astype(int)

    df['card1_txn_count'] = df.groupby('card1')['TransactionID'].transform('count')

    df['addr_mismatch'] = (
        df['addr1'].fillna(-1) != df['addr2'].fillna(-1)
    ).astype(int)

    df['is_large_txn'] = (df['TransactionAmt'] > 500).astype(int)

    c_cols   = [c for c in df.columns if c.startswith('C') and c[1:].isdigit()]
    df['C_sum'] = df[c_cols].fillna(0).sum(axis=1)

    df['is_mobile'] = (df.get('DeviceType','').fillna('') == 'mobile').astype(int)

    return df

train = engineer_features(train)
test  = engineer_features(test)

new_features = ['amt_log','amt_zscore','email_mismatch','P_email_risky',
                'hour','is_night','card1_txn_count','addr_mismatch',
                'is_large_txn','C_sum','is_mobile']

print(f"Engineered {len(new_features)} features:")
print(f"{'Feature':<24} {'Legit mean':>12} {'Fraud mean':>12}")
print("-" * 50)
for f in new_features:
    means = train.groupby('isFraud')[f].mean()
    print(f"  {f:<22} {means.get(0,0):>12.3f} {means.get(1,0):>12.3f}")

## Encode Categoricals & Save

Frequency encoding for categoricals — maps each category to how often it appears in training. Works well for tree models and handles unseen categories in test gracefully (map to 0).

In [ ]:
cat_cols = train.select_dtypes(include='object').columns.tolist()
cat_cols = [c for c in cat_cols if c not in ['TransactionID'] and c in test.columns]

print(f"Encoding {len(cat_cols)} categorical columns...")
for col in cat_cols:
    freq = train[col].value_counts()
    train[col] = train[col].map(freq).fillna(0)
    test[col]  = test[col].map(freq).fillna(0)

print("Done")

In [ ]:
import json

DROP = ['TransactionID', 'isFraud', 'hour', 'card1_amt_mean', 'card1_amt_std']
FEATURES = [c for c in train.columns if c not in DROP and c in test.columns]

X_train = train[FEATURES].fillna(-999)
y_train = train['isFraud']
X_test  = test[FEATURES].fillna(-999)

print(f"Features  : {len(FEATURES)}")
print(f"X_train   : {X_train.shape}")
print(f"X_test    : {X_test.shape}")
print(f"Fraud rate: {y_train.mean()*100:.2f}%")

X_train.to_parquet(OUT_DIR + 'X_train.parquet', index=False)
y_train.to_frame().to_parquet(OUT_DIR + 'y_train.parquet', index=False)
X_test.to_parquet(OUT_DIR + 'X_test.parquet', index=False)

with open(OUT_DIR + 'features.json', 'w') as f:
    json.dump(FEATURES, f)

print(f"\nSaved to {OUT_DIR}")